# Step 4: Convolutional Neural Network (CNN) Classifier

This notebook implements a CNN for MNIST - the proper way to handle image data.

**Key difference from MLP**: CNN preserves the 28×28 spatial structure and learns local patterns.

In [ ]:
import numpy as np
import struct
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow version: {tf.__version__}")
%matplotlib inline

## 1. Load Dataset

In [ ]:
def read_idx_images(filename):
    with open(filename, 'rb') as f:
        magic, num_images, rows, cols = struct.unpack('>IIII', f.read(16))
        images = np.fromfile(f, dtype=np.uint8).reshape(num_images, rows, cols)
    return images

def read_idx_labels(filename):
    with open(filename, 'rb') as f:
        magic, num_labels = struct.unpack('>II', f.read(8))
        labels = np.fromfile(f, dtype=np.uint8)
    return labels

In [ ]:
train_images = read_idx_images('../data/train-images.idx3-ubyte')
train_labels = read_idx_labels('../data/train-labels.idx1-ubyte')
test_images = read_idx_images('../data/t10k-images.idx3-ubyte')
test_labels = read_idx_labels('../data/t10k-labels.idx1-ubyte')

print(f"Training set: {train_images.shape}")
print(f"Test set: {test_images.shape}")

## 2. Data Preprocessing

**Key difference from MLP**: We keep the 28×28 spatial structure and add a channel dimension!

In [ ]:
X_train = train_images.astype('float32') / 255.0
X_test = test_images.astype('float32') / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

y_train = train_labels
y_test = test_labels

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Shape: (samples, height, width, channels)")
print(f"\nMLP would flatten to: {28*28} = 784 pixels (loses structure)")
print(f"CNN keeps: 28×28×1 (preserves spatial relationships)")

## 3. Build CNN Model

### Architecture

```
Input: 28×28×1 image
    ↓
Conv2D (32 filters, 3×3) → Detect edges
    ↓
MaxPooling (2×2) → Reduce size, keep important features
    ↓
Conv2D (64 filters, 3×3) → Detect shapes
    ↓
MaxPooling (2×2) → Reduce again
    ↓
Conv2D (64 filters, 3×3) → Detect complex patterns
    ↓
Flatten → Convert to 1D
    ↓
Dense (64) → Combine patterns
    ↓
Dropout (0.5) → Prevent overfitting
    ↓
Dense (10, softmax) → Output: probabilities for 0-9
```

In [ ]:
model = keras.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    layers.Conv2D(64, (3, 3), activation='relu'),
    
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

## 4. Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

### Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Model Evaluation

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Test Loss: {test_loss:.4f}")
print(f"Error Rate: {(1-test_accuracy)*100:.2f}%")
print(f"Misclassified: {int((1-test_accuracy)*10000)} out of 10,000")

In [ ]:
y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion Matrix (CNN)', fontsize=14)
plt.tight_layout()
plt.show()

### Classification Report

In [ ]:
print("Classification Report:")
print("=" * 60)
print(classification_report(y_test, y_pred_classes, digits=4))

## 6. Error Analysis

In [ ]:
misclassified_idx = np.where(y_pred_classes != y_test)[0]
print(f"Total misclassified: {len(misclassified_idx)} out of {len(y_test)}")

In [ ]:
sample_errors = misclassified_idx[:20] if len(misclassified_idx) >= 20 else misclassified_idx

fig, axes = plt.subplots(2, 10, figsize=(15, 3))
fig.suptitle('Misclassified Samples (CNN)', fontsize=14, color='red')

for i, idx in enumerate(sample_errors):
    ax = axes[i // 10, i % 10]
    ax.imshow(test_images[idx], cmap='gray')
    confidence = y_pred[idx][y_pred_classes[idx]] * 100
    ax.set_title(f'T:{y_test[idx]} P:{y_pred_classes[idx]}\n{confidence:.0f}%', fontsize=8, color='red')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Visualize What CNN Learns

### First Layer Filters (Edge Detectors)

In [ ]:
first_layer_weights = model.layers[0].get_weights()[0]
print(f"First conv layer filters shape: {first_layer_weights.shape}")
print(f"Each filter is 3×3 pixels")

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
fig.suptitle('32 Filters from First Conv Layer (Edge Detectors)', fontsize=14)

for i in range(32):
    ax = axes[i // 8, i % 8]
    ax.imshow(first_layer_weights[:, :, 0, i], cmap='gray')
    ax.set_title(f'#{i}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
sample_idx = 0
sample_image = X_test[sample_idx:sample_idx+1]

first_conv_layer = keras.Model(
    inputs=model.layers[0].input,
    outputs=model.layers[0].output
)
first_activation = first_conv_layer.predict(sample_image, verbose=0)

print(f"Sample digit: {y_test[sample_idx]}")
plt.figure(figsize=(3, 3))
plt.imshow(test_images[sample_idx], cmap='gray')
plt.title(f'Original Image (Digit: {y_test[sample_idx]})')
plt.axis('off')
plt.show()

## 8. Model Comparison Summary